In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
        
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score   
from sklearn.model_selection import cross_val_score

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df_train = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/train.csv", header = None)
trainLabels = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/trainLabels.csv", header = None)
df_test = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/test.csv", header = None)

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_train.describe()

In [ ]:
x=df_train
y=trainLabels.to_numpy()
x.head()

In [ ]:

print('train shape:', x.shape)
print('test shape:', df_test.shape)
print('trainLabel shape:', y.shape)

# Checking for missing data

In [ ]:
train_data = x
test_data = df_test
train_label = y


datasetHasNan = False
if train_data.count().min() == train_data.shape[0] and test_data.count().min() == test_data.shape[0] :
    print('There are no missing values.') 
else:
    datasetHasNan = True
    print('Yes, we have missing values')

# now list items    
if datasetHasNan == True:
    nas = pd.concat([train_data.isnull().sum(), test_data.isnull().sum()], axis=1, keys=['Train Dataset', 'Test Dataset']) 
    print('--'*40)
    print('Nan in the data sets')
    print(nas[nas.sum(axis=1) > 0])

# Preproccessing


## Train_Test split

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(train_data,train_label, test_size = 0.30, random_state = 101)
x_train.shape,x_test.shape,y_train.shape,y_test.shape

In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(x_train,y_train.ravel())
nb_predicted= model.predict(x_test)
acc_test_gnb=accuracy_score(y_test, nb_predicted)
print('Naive Bayes',acc_test_gnb)

#KNN
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(x_train,y_train.ravel())
knn_predicted= knn_model.predict(x_test)
acc_test_knn= accuracy_score(y_test,knn_predicted)
print('KNN',acc_test_knn)

#RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

rfc_model = RandomForestClassifier(n_estimators = 100,random_state = 22)
rfc_model.fit(x_train,y_train.ravel())
rfc_predicted = rfc_model.predict(x_test)
acc_test_rf= accuracy_score(y_test,rfc_predicted)
print('Random Forest',acc_test_rf)

#LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=0, solver='sag')
lr_model.fit(x_train,y_train.ravel())
lr_predicted = lr_model.predict(x_test)
acc_test_lr= accuracy_score(y_test, lr_predicted)
print('Logistic Regression',acc_test_lr)

#SVM
from sklearn.svm import SVC

svc_model = SVC(gamma = 'auto')
svc_model.fit(x_train,y_train.ravel())
svc_predicted = svc_model.predict(x_test)
acc_test_svm= accuracy_score(y_test, svc_predicted)
print('SVM',acc_test_svm)

#DECISON TREE
from sklearn.tree import DecisionTreeClassifier

dtree_model = DecisionTreeClassifier()
dtree_model.fit(x_train,y_train.ravel())
dtree_predicted = dtree_model.predict(x_test)
acc_test_dt= accuracy_score(y_test, dtree_predicted)
print('Decision Tree',acc_test_dt)

#XGBOOST
from xgboost import XGBClassifier

xgb = XGBClassifier()
xgb.fit(x_train,y_train.ravel())
xgb_predicted = xgb.predict(x_test)
acc_test_xg= accuracy_score(y_test, xgb_predicted)
print('XGBoost',acc_test_xg)

# comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')


acc_test =[acc_test_gnb,acc_test_knn,acc_test_dt,acc_test_rf,acc_test_svm,acc_test_lr,acc_test_xg]
title=["GNB","KNN","DT","RF","SVM","LR","XG"]
plt.figure(figsize=(10, 6))
sns.barplot(x=title, y=acc_test, palette='PiYG')


plt.title('Accuracy of Each Model')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.grid(True)

plt.show()

**KNN** gives the maximum accuracy after Feature Scaling.



# PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(0.85, whiten=True)
pca_train_data = pca.fit_transform(train_data)
print(pca_train_data.shape,'\n')

explained_variance = pca.explained_variance_ratio_ 
print(explained_variance)

In [ ]:
# NAIVE BAYES
from sklearn.naive_bayes import GaussianNB


nb_model = GaussianNB()
print('Naive Bayes',cross_val_score(nb_model,pca_train_data, train_label.ravel(), cv=10).mean())
cross_val_nb = cross_val_score(nb_model,pca_train_data, train_label.ravel(), cv=10).mean()

#KNN
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors = 5)

print('KNN',cross_val_score(knn_model,pca_train_data, train_label.ravel(), cv=10).mean())
cross_val_knn = cross_val_score(knn_model,pca_train_data, train_label.ravel(), cv=10).mean()

#RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

rfc_model = RandomForestClassifier(n_estimators = 100,random_state = 99)

print('Random Forest',cross_val_score(rfc_model,pca_train_data, train_label.ravel(), cv=10).mean())
cross_val_rf = cross_val_score(rfc_model,pca_train_data, train_label.ravel(), cv=10).mean()

#SVM
from sklearn.svm import SVC

svc_model = SVC(gamma = 'auto')
cross_val_svm = cross_val_score(svc_model,pca_train_data, train_label.ravel(), cv=10).mean()
print('SVM',cross_val_score(svc_model,pca_train_data, train_label.ravel(), cv=10).mean())


#DECISION TREE
from sklearn.tree import DecisionTreeClassifier

dtree_model = DecisionTreeClassifier()
cross_val_dt = cross_val_score(dtree_model,pca_train_data, train_label.ravel(), cv=10).mean()
print('Decision Tree',cross_val_score(dtree_model,pca_train_data, train_label.ravel(), cv=10).mean())


#XGBOOST
from xgboost import XGBClassifier

xgb = XGBClassifier()
cross_val_xg = cross_val_score(xgb,pca_train_data, train_label.ravel(), cv=10).mean()
print('XGBoost',cross_val_score(xgb,pca_train_data, train_label.ravel(), cv=10).mean())

In [ ]:
cross_val_test =[cross_val_nb,cross_val_knn,cross_val_dt,cross_val_rf,cross_val_svm,cross_val_xg]
title=["GNB","KNN","DT","RF","SVM","XG"]
colors={'grey',"red","blue","green","pink","orange","yellow"}
plt.bar(title, cross_val_test,color=colors)
plt.grid()
plt.show()

# Applying Gaussian Mixture and Grid Search to improve the accuracy

 We select the above three algorithms **(KNN, Random Forest and SVM)** which gave **maximum accuracy** for further analysis

In [ ]:
# Importing libraries
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC

X = np.r_[train_data,test_data]
print('X shape :',X.shape)
print('\n')

# USING THE GAUSSIAN MIXTURE MODEL 

#The Bayesian information criterion (BIC) can be used to select the number of components in a Gaussian Mixture in an efficient way. 
#In theory, it recovers the true number of components only in the asymptotic regime
lowest_bic = np.infty
bic = []
n_components_range = range(1, 7)

#The GaussianMixture comes with different options to constrain the covariance of the difference classes estimated: 
# spherical, diagonal, tied or full covariance.
cv_types = ['spherical', 'tied', 'diag', 'full']
for cv_type in cv_types:
    for n_components in n_components_range:
        gmm = GaussianMixture(n_components=n_components,covariance_type=cv_type)
        gmm.fit(X)
        bic.append(gmm.aic(X))
        if bic[-1] < lowest_bic:
            lowest_bic = bic[-1]
            best_gmm = gmm
            
best_gmm.fit(X)
gmm_train = best_gmm.predict_proba(train_data)
gmm_test = best_gmm.predict_proba(test_data)

# Now Applying Grid Search Algorithm:

To identify the **best algorithm** and **best parameters**

In [ ]:
##### ------------------------ Random Forest Classifier -------------------------- #####
rfc = RandomForestClassifier(random_state=99)

#USING GRID SEARCH
#The first step you need to perform is to create a dictionary of all the parameters and their corresponding set of values that you want to test for best performance. 
n_estimators = [10, 50, 100, 200,400]
max_depth = [3, 10, 20, 40]
param_grid = dict(n_estimators=n_estimators,max_depth=max_depth)
#in the above script we want to find which value (out of 10, 50, 100, 200,400) provides the highest accuracy.
#The Grid Search algorithm basically tries all possible combinations of parameter values and returns the combination with the highest accuracy.

#create an instance of the GridSearchCV class
grid_search_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv = 10,scoring='accuracy',n_jobs=-1).fit(gmm_train, train_label.ravel())
#Using 10-folds CV whereas a value of -1 for n_jobs parameter means that use all available computing power.

rfc_best = grid_search_rfc.best_estimator_
print('Random Forest Best Score',grid_search_rfc.best_score_)
print('Random Forest Best Parmas',grid_search_rfc.best_params_)
print('Random Forest Accuracy',cross_val_score(rfc_best,gmm_train, train_label.ravel(), cv=10).mean())
print('--'*40,'\n')

##### -------------------------- KNN -------------------------- ##### 
knn = KNeighborsClassifier()

#USING GRID SEARCH
#First off, the n_neighbors should always be an odd number. You can choose an even number, but in the case of a tie vote, 
#the decision on which class to assign will be done randomly when weights is set to uniform. By choosing an odd number, there are no ties.
n_neighbors=[3,5,6,7,8,9,10]
param_grid = dict(n_neighbors=n_neighbors)
# Another important thing to note is that when you do a GridSearch, you’re running many more models than when you simply fit and score. It’s important to set 
# verbose so you’ll get feedback on the model and know how long it may take to finish. 
# kNN can take a long time to complete as it measures the individual distances for each point in the test set.

#create an instance of the GridSearchCV class
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_grid, cv = 10, n_jobs=-1,scoring='accuracy').fit(gmm_train,train_label.ravel())
knn_best = grid_search_knn.best_estimator_
print('KNN Best Score', grid_search_knn.best_score_)
print('KNN Best Params',grid_search_knn.best_params_)
print('KNN Accuracy',cross_val_score(knn_best,gmm_train, train_label.ravel(), cv=10).mean())
print('--'*40,'\n')

##### -------------------------- SVM -------------------------- ##### 
svc = SVC()

#USING GRID SEARCH
#setup a parameter grid (using multiples of 10’s is a good place to start) and then pass the algorithm, parameter grid and number of cross validations to the GridSearchCV method.
parameters = [{'kernel':['linear'],'C':[1,10,100]},
              {'kernel':['rbf'],'C':[1,10,100],'gamma':[0.05,0.0001,0.01,0.001]}]

#create an instance of the GridSearchCV class
grid_search_svm = GridSearchCV(estimator=svc, param_grid=parameters, cv = 10, n_jobs=-1,scoring='accuracy').fit(gmm_train, train_label.ravel())
svm_best = grid_search_svm.best_estimator_
print('SVM Best Score',grid_search_svm.best_score_)
print('SVM Best Params',grid_search_svm.best_params_)
print('SVM Accuracy',cross_val_score(svm_best,gmm_train, train_label.ravel(), cv=10).mean())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# داده‌ها (تغییر دهید به داده‌های خودتان)
# gmm_train, train_label

# تعریف مدل و پارامترها
knn = KNeighborsClassifier()
param_grid = {'n_neighbors': np.arange(1, 26)}

# اجرای GridSearchCV
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_grid, cv=10, n_jobs=-1, scoring='accuracy')
grid_search_knn.fit(gmm_train, train_label.ravel())

# بهترین مدل و پارامترها
knn_best = grid_search_knn.best_estimator_
print('KNN Best Score', grid_search_knn.best_score_)
print('KNN Best Params', grid_search_knn.best_params_)
print('KNN Accuracy', cross_val_score(knn_best, gmm_train, train_label.ravel(), cv=10).mean())
print('--' * 40, '\n')

# رسم نمودار
results = grid_search_knn.cv_results_
mean_test_scores = results['mean_test_score']
param_k = results['param_n_neighbors']

plt.figure(figsize=(10, 6))
plt.plot(param_k, mean_test_scores, marker='o',color='purple')

plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean Validation Accuracy')
plt.title('Validation Accuracy vs. Number of Neighbors')
plt.grid(True)
plt.show()


<p style="color:purple; font-size:25px;">The results from the GridSearchCV and plot indicate that the optimal number of neighbors (k) for the KNN model is 3, with a best score of 0.996.</p>

In [ ]:
# Fitting our model
rfc_best.fit(gmm_train,train_label.ravel())
pred  = rfc_best.predict(gmm_test)
rfc_best_pred = pd.DataFrame(pred)

rfc_best_pred.index += 1

# FRAMING OUR SOLUTION
rfc_best_pred.columns = ['Solution']
rfc_best_pred['Id'] = np.arange(1,rfc_best_pred.shape[0]+1)
rfc_best_pred = rfc_best_pred[['Id', 'Solution']]

rfc_best_pred.to_csv('Submission.csv',index=False)